# s8 - How I would approach this in 2026 (addendum)

> **This is a forward-looking addendum, written in 2026.** The original study
> (`s1`-`s7`) was completed in 2024 using the GAN-based methods that were
> state-of-the-art at the time, and its results stand on their own. Nothing in
> the original work has been changed or re-run. This notebook simply records how
> the field has moved on and what I would do differently if I started today.

## What the 2024 study found

Training an LSTM on synthetic data and testing it on **real** GOOG prices:

| Generator | RMSE (synthetic only) | RMSE (synthetic + real) |
| --- | :---: | :---: |
| **TimeGAN** | **4.18** | 11.75 |
| WGAN-GP | 22.93 | 17.82 |
| DoppelGANger | 31.99 | **8.85** |
| CTGAN | 38.78 | 44.30 |

TimeGAN was best for standalone training utility; DoppelGANger best as a
real-data augmenter; CTGAN (a tabular method) was poorly suited to time series.
A recurring theme: a generator's distributional realism (PCA/t-SNE) did not
predict its downstream forecasting utility.

## 1. Generators: diffusion has overtaken GANs for time series

Since 2024, denoising-diffusion and score-based models have become the strongest
general approach for time-series generation, addressing two pain points seen in
this project: unstable adversarial training and mode collapse (visible in the
CTGAN/WGAN-GP outputs).

Methods I would benchmark today:
- **Diffusion-TS** - interpretable diffusion for general time-series generation.
- **TSDiff** - self-guided diffusion, strong for forecasting-oriented synthesis.
- **TimeGAN / DoppelGANger** kept as GAN baselines for a like-for-like comparison
  with the 2024 results.

Diffusion models are slower to sample but far more stable to train, and they tend
to preserve temporal dynamics better than the adversarial methods used here.

## 2. Evaluation: quantify fidelity, do not eyeball it

The 2024 study judged distribution overlap with PCA and t-SNE plots. Today I would
add a quantitative suite so generators can be ranked objectively:
- **Predictive score** - train on synthetic, test on real (the metric this project
  already uses); keep it as the headline.
- **Discriminative score** - train a classifier to tell real from synthetic; closer
  to 50% accuracy is better.
- **Distributional distances** - e.g. sliced-Wasserstein / MMD between real and
  synthetic feature distributions.
- **Temporal-dependency checks** - compare autocorrelation structure, not just
  marginal distributions.

## 3. Downstream model: replace the LSTM baseline

The LSTM was a reasonable 2024 baseline. As the downstream forecaster I would now
use a modern architecture and re-run the same train-on-synthetic / test-on-real
protocol:
- **PatchTST** - patch-based transformer, strong long-horizon forecaster.
- **N-HiTS / N-BEATS** - efficient, accurate MLP-based forecasters.
- A **statistical baseline** (e.g. seasonal naive / ARIMA) to anchor the metrics -
  something the original study lacked.

## 4. A concrete plan if I redid this

1. Keep `s1`'s dataset and feature engineering (still sound).
2. Generators: Diffusion-TS and TSDiff, plus TimeGAN/DoppelGANger as baselines.
3. Evaluation: predictive + discriminative + distributional + temporal metrics.
4. Downstream forecasters: PatchTST and N-HiTS, with a seasonal-naive anchor.
5. Report a single results table ranking each generator by predictive score, and
   test whether the 2024 finding holds - that realism does not equal utility.

### Selected references
- Yoon et al., *Time-series Generative Adversarial Networks* (TimeGAN), 2019.
- Lin et al., *Using GANs for Sharing Networked Time Series Data* (DoppelGANger), 2020.
- Yuan & Qiao, *Diffusion-TS: Interpretable Diffusion for General Time Series Generation*, 2024.
- Kollovieh et al., *Predict, Refine, Synthesize: Self-Guiding Diffusion (TSDiff)*, 2023.
- Nie et al., *A Time Series is Worth 64 Words: Long-term Forecasting with Transformers (PatchTST)*, 2023.
- Challu et al., *N-HiTS: Neural Hierarchical Interpolation for Time Series Forecasting*, 2023.